# Hurricane Katrina — intensity timeline (2005)

Pull the North Atlantic best track around late August 2005, isolate Hurricane Katrina, and chart its wind
and pressure over time. Point-mode query against the HURDAT2 reanalysis via the `tropycal` backend.
See [Usage](../../reference/tropycal/usage.md).

## Setup

Consolidate the imports up front. `earthlens` provides the unified `EarthLens` entry point; `matplotlib`
draws the intensity timeline. We also prepare an output directory for the best-track features.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from earthlens.core import EarthLens

OUT_DIR = Path('tropycal_output')
OUT_DIR.mkdir(exist_ok=True)

## 1 · Best-track query

`tropycal` is a *vector* backend, so we build the request first — source, basin (`variables`), date window,
and a bounding box over the Gulf of Mexico and western Atlantic. Keeping the construction on its own line
makes the request easy to read and re-run.

In [ ]:
track = EarthLens(
    variables=['north_atlantic'],
    data_source='tropycal',
    start='2005-08-23',
    end='2005-08-31',
    aoi=[-95.0, 15.0, -60.0, 35.0],
    source='hurdat',
    path=str(OUT_DIR),
)

### Download the fixes

`download()` fetches the best-track fixes in the window. The call is wrapped in a try/except so the
notebook stays safe under nbval-lax when run offline or without the `[tropycal]` extra.

In [ ]:
fixes = None
try:
    fixes = track.download(progress_bar=False)
    print(len(fixes), 'fixes in window')
except Exception as exc:
    print(f'skipped live query: {type(exc).__name__}: {exc}')

## 2 · Isolate Hurricane Katrina

Filter the fixes to the `KATRINA` track, sort them in time, and report the peak intensity (the strongest
wind fix and its pressure/category).

In [ ]:
if fixes is not None and len(fixes):
    katrina = fixes[fixes['name'] == 'KATRINA'].sort_values('time')
    peak = katrina.loc[katrina['vmax_kt'].idxmax()]
    print(
        f"Katrina: {len(katrina)} fixes, peak {peak['vmax_kt']:.0f} kt / "
        f"{peak['mslp_hpa']:.0f} hPa (cat {int(peak['category'])})"
    )

## 3 · Intensity timeline

Chart maximum sustained wind and minimum sea-level pressure over time on a shared time axis (twin y-axes),
so the classic wind-up / pressure-down signature of an intensifying hurricane is visible at a glance.

In [ ]:
if fixes is not None and len(fixes):
    katrina = fixes[fixes['name'] == 'KATRINA'].sort_values('time')
    fig, ax1 = plt.subplots(figsize=(9, 5))
    ax1.plot(katrina['time'], katrina['vmax_kt'], 'o-', color='crimson')
    ax1.set_ylabel('Max sustained wind (kt)', color='crimson')
    ax2 = ax1.twinx()
    ax2.plot(katrina['time'], katrina['mslp_hpa'], 's--', color='navy')
    ax2.set_ylabel('Min sea-level pressure (hPa)', color='navy')
    ax1.set_title('Hurricane Katrina (2005) intensity timeline')
    ax1.set_xlabel('Time (UTC)')
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()